### ЗАДАЧА: Операционный отчет по доставкам за смену

Логистическая команда в конце смены получает журнал доставок.
Нужно подготовить:
- `JSON`-отчет для операционного дашборда,
- `pickle`-снимок всех обработанных данных для внутреннего разбора.

НЕОБХОДИМО:

1. Преобразовать строки из `rows` в список словарей `deliveries`.
2. Для каждого заказа привести типы:
   - `planned_minutes` -> `int`
   - `actual_minutes` -> `int`
   - `order_value` -> `float`
3. Для каждой доставки вычислить поле `delay_minutes`:
   - если `actual_minutes > planned_minutes`, то разница,
   - иначе `0`.
4. Построить словарь `courier_summary`, где по каждому курьеру будет:
   - `deliveries_count`
   - `completed_count`
   - `returned_count`
   - `late_count`
   - `total_revenue`
5. Построить словарь `zone_summary`, где по каждой зоне будет:
   - `deliveries_count`
   - `late_count`
   - `avg_delay_minutes`
6. Собрать список `incident_deliveries`, если:
   - `status == 'returned'`,
   - или `delay_minutes >= 20`.
7. Для `incident_deliveries` оставить поля:
   - `delivery_id`
   - `courier`
   - `zone`
   - `status`
   - `delay_minutes`
   - `order_value`
8. Найти курьера с наибольшей выручкой и сохранить в `top_courier`.
9. Отсортировать `incident_deliveries` по `delay_minutes` по убыванию.
10. Округлить денежные значения до 2 знаков.
11. Собрать `public_report` со структурой:
   - `report_name`
   - `deliveries_count`
   - `top_courier`
   - `courier_summary`
   - `zone_summary`
   - `incident_deliveries`
12. Сохранить `public_report` в `delivery_operations_report.json`.
13. Сохранить внутренний снимок в `delivery_operations_snapshot.pkl`.
14. Прочитать оба файла обратно и вывести результаты.


In [1]:
import json
import pickle


# row format: delivery_id|courier|zone|status|planned_minutes|actual_minutes|order_value
rows = [
    "DL-701|Sergey|center|completed|35|33|1250.00",
    "DL-702|Ira|north|completed|40|68|890.50",
    "DL-703|Sergey|center|returned|25|30|540.00",
    "DL-704|Nikita|south|completed|50|49|2310.90",
    "DL-705|Ira|north|completed|30|57|760.40",
    "DL-706|Nikita|south|completed|45|72|1890.00",
    "DL-707|Sergey|west|completed|20|20|420.00",
    "DL-708|Ira|north|returned|35|41|1100.00",
]

deliveries = []
courier_summary = {}
zone_summary = {}
incident_deliveries = []

for row in rows:
    delivery_id, courier, zone, status, planned_raw, actual_raw, order_value_raw = row.split("|")

    # TODO 1: преобразуйте planned_raw в int и сохраните в planned_minutes
    # TODO 2: преобразуйте actual_raw в int и сохраните в actual_minutes
    # TODO 3: преобразуйте order_value_raw в float и сохраните в order_value

    # TODO 4: вычислите delay_minutes
    planned_minutes = int(planned_raw)
    actual_minutes = int(actual_raw)
    order_value = float(order_value_raw)

    delay_minutes = actual_minutes - planned_minutes if actual_minutes > planned_minutes else 0

    delivery = {
        "delivery_id": delivery_id,
        "courier": courier,
        "zone": zone,
        "status": status,
        # TODO 5: добавьте planned_minutes
        # TODO 6: добавьте actual_minutes
        # TODO 7: добавьте order_value
        # TODO 8: добавьте delay_minutes
        "planned_minutes": planned_minutes,
        "actual_minutes": actual_minutes,
        "order_value": round(order_value, 2),
        "delay_minutes": delay_minutes,
    }
    deliveries.append(delivery)

    # TODO 9: если courier еще отсутствует в courier_summary,
    # создайте для него словарь с полями:
    #   deliveries_count -> 0
    #   completed_count -> 0
    #   returned_count -> 0
    #   late_count -> 0
    #   total_revenue -> 0.0
    if courier not in courier_summary:
        courier_summary[courier] = {
            "deliveries_count": 0,
            "completed_count": 0,
            "returned_count": 0,
            "late_count": 0,
            "total_revenue": 0.0,
        }

    # TODO 10: увеличьте deliveries_count на 1
    # TODO 11: если status == 'completed', увеличьте completed_count и прибавьте order_value к total_revenue
    # TODO 12: если status == 'returned', увеличьте returned_count
    # TODO 13: если delay_minutes > 0, увеличьте late_count
    courier_summary[courier]["deliveries_count"] += 1

    if status == "completed":
        courier_summary[courier]["completed_count"] += 1
        courier_summary[courier]["total_revenue"] += order_value

    if status == "returned":
        courier_summary[courier]["returned_count"] += 1

    if delay_minutes > 0:
        courier_summary[courier]["late_count"] += 1

    # TODO 14: если zone еще отсутствует в zone_summary,
    # создайте для нее словарь с полями:
    #   deliveries_count -> 0
    #   late_count -> 0
    #   total_delay_minutes -> 0
    if zone not in zone_summary:
        zone_summary[zone] = {
            "deliveries_count": 0,
            "late_count": 0,
            "total_delay_minutes": 0,
        }

    # TODO 15: увеличьте deliveries_count по зоне
    # TODO 16: прибавьте delay_minutes к total_delay_minutes
    # TODO 17: если delay_minutes > 0, увеличьте late_count по зоне
    zone_summary[zone]["deliveries_count"] += 1
    zone_summary[zone]["total_delay_minutes"] += delay_minutes

    if delay_minutes > 0:
        zone_summary[zone]["late_count"] += 1

    # TODO 18: если доставка инцидентная,
    # добавьте в incident_deliveries словарь только с нужными полями
    if status == "returned" or delay_minutes >= 20:
        incident_deliveries.append({
            "delivery_id": delivery_id,
            "courier": courier,
            "zone": zone,
            "status": status,
            "delay_minutes": delay_minutes,
            "order_value": round(order_value, 2),
        })

top_courier = None
top_revenue = 0

for courier, summary in courier_summary.items():
    # TODO 19: сравните total_revenue с top_revenue и при необходимости обновите top_courier
    if summary["total_revenue"] > top_revenue:
        top_revenue = summary["total_revenue"]
        top_courier = courier

for zone, summary in zone_summary.items():
    # TODO 20: вычислите avg_delay_minutes = total_delay_minutes / deliveries_count
    # TODO 21: округлите avg_delay_minutes до 2 знаков
    # TODO 22: удалите из summary служебное поле total_delay_minutes
    avg_delay = summary["total_delay_minutes"] / summary["deliveries_count"]
    summary["avg_delay_minutes"] = round(avg_delay, 2)
    del summary["total_delay_minutes"]

# TODO 23: округлите total_revenue у всех курьеров до 2 знаков
for summary in courier_summary.values():
    summary["total_revenue"] = round(summary["total_revenue"], 2)

# TODO 24: отсортируйте incident_deliveries по delay_minutes по убыванию
incident_deliveries.sort(key=lambda x: x["delay_minutes"], reverse=True)

public_report = {
        "report_name": "shift_delivery_operations",
        "deliveries_count": len(deliveries),
        # TODO 25: добавьте top_courier
        # TODO 26: добавьте courier_summary
        # TODO 27: добавьте zone_summary
        # TODO 28: добавьте incident_deliveries
        "top_courier": top_courier,
        "courier_summary": courier_summary,
        "zone_summary": zone_summary,
        "incident_deliveries": incident_deliveries,
    }

snapshot = {
    "rows": rows,
    "deliveries": deliveries,
    "courier_summary": courier_summary,
    "zone_summary": zone_summary,
    "incident_deliveries": incident_deliveries,
    "top_courier": top_courier,
}

# TODO 29: сохраните public_report в delivery_operations_report.json через json.dump
#   используйте ensure_ascii=False и indent=2

# TODO 30: сохраните snapshot в delivery_operations_snapshot.pkl через pickle.dump

# TODO 31: прочитайте delivery_operations_report.json в loaded_report через json.load
# TODO 32: прочитайте delivery_operations_snapshot.pkl в loaded_snapshot через pickle.load
with open("delivery_operations_report.json", "w", encoding="utf-8") as f:
    json.dump(public_report, f, ensure_ascii=False, indent=2)

with open("delivery_operations_snapshot.pkl", "wb") as f:
    pickle.dump(snapshot, f)

with open("delivery_operations_report.json", "r", encoding="utf-8") as f:
    loaded_report = json.load(f)

with open("delivery_operations_snapshot.pkl", "rb") as f:
    loaded_snapshot = pickle.load(f)

print("Доставки:")
print(deliveries)
print()

print("Сводка по курьерам:")
print(courier_summary)
print()

print("Сводка по зонам:")
print(zone_summary)
print()

print("Инцидентные доставки:")
print(incident_deliveries)
print()

print("Отчет:")
print(public_report)
print()

print("Данные из JSON:")
print(loaded_report)
print()

print("Данные из pickle:")
print(loaded_snapshot)


Доставки:
[{'delivery_id': 'DL-701', 'courier': 'Sergey', 'zone': 'center', 'status': 'completed', 'planned_minutes': 35, 'actual_minutes': 33, 'order_value': 1250.0, 'delay_minutes': 0}, {'delivery_id': 'DL-702', 'courier': 'Ira', 'zone': 'north', 'status': 'completed', 'planned_minutes': 40, 'actual_minutes': 68, 'order_value': 890.5, 'delay_minutes': 28}, {'delivery_id': 'DL-703', 'courier': 'Sergey', 'zone': 'center', 'status': 'returned', 'planned_minutes': 25, 'actual_minutes': 30, 'order_value': 540.0, 'delay_minutes': 5}, {'delivery_id': 'DL-704', 'courier': 'Nikita', 'zone': 'south', 'status': 'completed', 'planned_minutes': 50, 'actual_minutes': 49, 'order_value': 2310.9, 'delay_minutes': 0}, {'delivery_id': 'DL-705', 'courier': 'Ira', 'zone': 'north', 'status': 'completed', 'planned_minutes': 30, 'actual_minutes': 57, 'order_value': 760.4, 'delay_minutes': 27}, {'delivery_id': 'DL-706', 'courier': 'Nikita', 'zone': 'south', 'status': 'completed', 'planned_minutes': 45, 'actu